# Семинар 2. Исследование методов линейной регрессии

#### Критерий оценивания:
#### Пункт 14: максимум 0,5 балла
#### Пункт 15: максимум 0,5 балла

#### Итого за работу: максимум 1 балл
#### P.S. пункты 14 и 15 без пунктов 1-13 не засчитываются, ответы на вопросы из ИИ не засчитываются

1. Загрузите датасет и выведите на экран первые несколько строк

In [1]:
import pandas as pd

# Загрузка данных
df = pd.read_csv('auto_dataset.csv')

# Вывод первых 5 строк
df.head()

,brand,model,vehicleType,gearbox,fuelType,notRepairedDamage,powerPS,kilometer,autoAgeMonths,price
0,volkswagen,golf,kleinwagen,manuell,benzin,nein,75,150000,177,1500
1,skoda,fabia,kleinwagen,manuell,diesel,nein,69,90000,93,3600
2,bmw,3er,limousine,manuell,benzin,ja,102,150000,246,650
3,peugeot,2_reihe,cabrio,manuell,benzin,nein,109,150000,140,2200
4,mazda,3_reihe,limousine,manuell,benzin,nein,105,150000,136,2000


2. Разбейте выборку на признаки и ответы. Закодируйте категориальные признаки.

In [2]:
# 1. Отделяем ответы (y) и признаки (X)
y = df['price']
X = df.drop(columns=['price'])

# 2. Закодируем категориальные признаки с помощью One-Hot Encoding
# drop_first=True удаляет один из столбцов каждой категории, избегая мультиколлинеарности
X_encoded = pd.get_dummies(X, drop_first=True)

# Посмотрим на размерность полученной матрицы признаков и первые строки
print(f"Размерность признаков: {X_encoded.shape}")
X_encoded.head()

Размерность признаков: (1000, 204)


,powerPS,kilometer,autoAgeMonths,brand_audi,brand_bmw,brand_chevrolet,brand_chrysler,brand_citroen,brand_dacia,brand_daewoo,...,vehicleType_kleinwagen,vehicleType_kombi,vehicleType_limousine,vehicleType_suv,gearbox_manuell,fuelType_benzin,fuelType_diesel,fuelType_hybrid,fuelType_lpg,notRepairedDamage_nein
0,75,150000,177,False,False,False,False,False,False,False,...,True,False,False,False,True,True,False,False,False,True
1,69,90000,93,False,False,False,False,False,False,False,...,True,False,False,False,True,False,True,False,False,True
2,102,150000,246,False,True,False,False,False,False,False,...,False,False,True,False,True,True,False,False,False,False
3,109,150000,140,False,False,False,False,False,False,False,...,False,False,False,False,True,True,False,False,False,True
4,105,150000,136,False,False,False,False,False,False,False,...,False,False,True,False,True,True,False,False,False,True


3. Разбейте датасет на train val test в отношении 8:1:1

In [4]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Сначала делим на train (80%) и временную выборку temp (20%)
X_train_raw, X_temp, y_train_raw, y_temp = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# 2. Временную выборку делим пополам: 10% на val и 10% на test
X_val_raw, X_test_raw, y_val_raw, y_test_raw = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# 3. Масштабируем числовые признаки (fit делаем только на train!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)
X_test_scaled = scaler.transform(X_test_raw)

# 4. Добавляем единичный столбец (bias / свободный член)
X_train = np.hstack([np.ones((X_train_scaled.shape[0], 1)), X_train_scaled])
X_val = np.hstack([np.ones((X_val_scaled.shape[0], 1)), X_val_scaled])
X_test = np.hstack([np.ones((X_test_scaled.shape[0], 1)), X_test_scaled])

# Приводим y к numpy массивам для векторных вычислений
y_train = y_train_raw.values.astype(float)
y_val = y_val_raw.values.astype(float)
y_test = y_test_raw.values.astype(float)

print(f"Размер train: {X_train.shape[0]} объектов ({X_train.shape[0]/len(df):.0%})")
print(f"Размер val:   {X_val.shape[0]} объектов ({X_val.shape[0]/len(df):.0%})")
print(f"Размер test:  {X_test.shape[0]} объектов ({X_test.shape[0]/len(df):.0%})")

Размер train: 800 объектов (80%)
Размер val:   100 объектов (10%)
Размер test:  100 объектов (10%)


4. Исследуйте VGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [5]:
# Функции расчета метрик ошибки MSE и коэффициента детерминации R^2
def compute_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def compute_r2(y_true, y_pred):
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    ss_res = np.sum((y_true - y_pred) ** 2)
    return 1 - (ss_res / (ss_tot + 1e-12))

# Обучение Vanilla Gradient Descent (VGD) с постоянным шагом eta
def train_vgd_const(X, y, eta, max_iter=1000, tol=1e-5):
    l = X.shape[0]
    w = np.zeros(X.shape[1])
    iters = 0
    for k in range(1, max_iter + 1):
        iters = k
        w_prev = w.copy()
        # Градиент MSE по всей выборке: 2/l * X.T @ (Xw - y)
        grad = (2.0 / l) * (X.T @ (X @ w - y))
        w -= eta * grad
        if np.linalg.norm(w - w_prev) < tol:
            break
    return w, iters

# Сетка шагов от 10^-5 до 1
grid_n = np.logspace(-5, 0, 6)

best_loss_val_vgd = float('inf')
best_n_vgd = None
best_train_loss_vgd = None
best_train_r2_vgd = None

print("=== Перебор шага n для VGD ===")
for n in grid_n:
    w, iters = train_vgd_const(X_train, y_train, eta=n)
    
    pred_train = X_train @ w
    pred_val = X_val @ w
    
    loss_tr = compute_loss(y_train, pred_train)
    r2_tr = compute_r2(y_train, pred_train)
    loss_v = compute_loss(y_val, pred_val)
    
    print(f"n = {n:7.5f} | Train Loss: {loss_tr:12.2f} | R2_train: {r2_tr:7.4f} | Val Loss: {loss_v:12.2f}")
    
    # Отсекаем расходимость (NaN / inf)
    if not np.isnan(loss_v) and not np.isinf(loss_v):
        if loss_v < best_loss_val_vgd:
            best_loss_val_vgd = loss_v
            best_n_vgd = n
            best_train_loss_vgd = loss_tr
            best_train_r2_vgd = r2_tr

# Финальное тестирование на test с наилучшим n
best_w_vgd, test_iters_vgd = train_vgd_const(X_train, y_train, eta=best_n_vgd)
pred_test_vgd = X_test @ best_w_vgd
loss_test_vgd = compute_loss(y_test, pred_test_vgd)
r2_test_vgd = compute_r2(y_test, pred_test_vgd)

print("\n--- Результаты VGD с наилучшим постоянным шагом ---")
print(f"Лучший шаг n:       {best_n_vgd}")
print(f"Loss_train:         {best_train_loss_vgd:.2f}")
print(f"R^2_train:          {best_train_r2_vgd:.4f}")
print(f"Loss_val:           {best_loss_val_vgd:.2f}")
print(f"Loss_test:          {loss_test_vgd:.2f}")
print(f"R^2_test:           {r2_test_vgd:.4f}")
print(f"Число итераций:     {test_iters_vgd}")

=== Перебор шага n для VGD ===
n = 0.00001 | Train Loss: 103219957.96 | R2_train: -0.6960 | Val Loss: 100795835.25
n = 0.00010 | Train Loss:  66140165.62 | R2_train: -0.0867 | Val Loss:  62140873.99
n = 0.00100 | Train Loss:  14705040.17 | R2_train:  0.7584 | Val Loss:  15283398.44
n = 0.01000 | Train Loss:  12508110.96 | R2_train:  0.7945 | Val Loss:  17329793.31
n = 0.10000 | Train Loss:  12044638.89 | R2_train:  0.8021 | Val Loss:  17791469.14
n = 1.00000 | Train Loss:          nan | R2_train:     nan | Val Loss:          nan

--- Результаты VGD с наилучшим постоянным шагом ---
Лучший шаг n:       0.001
Loss_train:         14705040.17
R^2_train:          0.7584
Loss_val:           15283398.44
Loss_test:          33196931.61
R^2_test:           0.5240
Число итераций:     1000


C:\Users\kirill\AppData\Local\Temp\ipykernel_10680\3140609951.py:19: RuntimeWarning: overflow encountered in matmul
  grad = (2.0 / l) * (X.T @ (X @ w - y))
C:\Users\kirill\AppData\Local\Temp\ipykernel_10680\3140609951.py:19: RuntimeWarning: invalid value encountered in matmul
  grad = (2.0 / l) * (X.T @ (X @ w - y))


5. Исследуйте VGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [6]:
# Обучение Vanilla Gradient Descent (VGD) с переменным шагом TimeDecayLR
def train_vgd_decay(X, y, lam, max_iter=1000, tol=1e-5, s0=1.0, p=0.5):
    l = X.shape[0]
    w = np.zeros(X.shape[1])
    iters = 0
    for k in range(1, max_iter + 1):
        iters = k
        w_prev = w.copy()
        
        # Шаг обучения по формуле TimeDecayLR из лекции
        eta_k = lam * ((s0 / (s0 + k)) ** p)
        
        # Градиент MSE по всей обучающей выборке
        grad = (2.0 / l) * (X.T @ (X @ w - y))
        w -= eta_k * grad
        
        if np.linalg.norm(w - w_prev) < tol:
            break
    return w, iters

# Сетка параметра lambda от 10^-5 до 1
grid_lambda = np.logspace(-5, 0, 6)

best_loss_val_vgd_decay = float('inf')
best_lam_vgd = None
best_train_loss_vgd_decay = None
best_train_r2_vgd_decay = None

print("=== Перебор параметра lambda (TimeDecayLR) для VGD ===")
for lam in grid_lambda:
    w, iters = train_vgd_decay(X_train, y_train, lam=lam)
    
    pred_train = X_train @ w
    pred_val = X_val @ w
    
    loss_tr = compute_loss(y_train, pred_train)
    r2_tr = compute_r2(y_train, pred_train)
    loss_v = compute_loss(y_val, pred_val)
    
    print(f"lambda = {lam:7.5f} | Train Loss: {loss_tr:12.2f} | R2_train: {r2_tr:7.4f} | Val Loss: {loss_v:12.2f}")
    
    if not np.isnan(loss_v) and not np.isinf(loss_v):
        if loss_v < best_loss_val_vgd_decay:
            best_loss_val_vgd_decay = loss_v
            best_lam_vgd = lam
            best_train_loss_vgd_decay = loss_tr
            best_train_r2_vgd_decay = r2_tr

# Финальное тестирование на test с наилучшим lambda
best_w_vgd_decay, test_iters_vgd_decay = train_vgd_decay(X_train, y_train, lam=best_lam_vgd)
pred_test_vgd_decay = X_test @ best_w_vgd_decay
loss_test_vgd_decay = compute_loss(y_test, pred_test_vgd_decay)
r2_test_vgd_decay = compute_r2(y_test, pred_test_vgd_decay)

print("\n--- Результаты VGD с TimeDecayLR ---")
print(f"Лучший lambda:      {best_lam_vgd}")
print(f"Loss_train:         {best_train_loss_vgd_decay:.2f}")
print(f"R^2_train:          {best_train_r2_vgd_decay:.4f}")
print(f"Loss_val:           {best_loss_val_vgd_decay:.2f}")
print(f"Loss_test:          {loss_test_vgd_decay:.2f}")
print(f"R^2_test:           {r2_test_vgd_decay:.4f}")
print(f"Число итераций:     {test_iters_vgd_decay}")

=== Перебор параметра lambda (TimeDecayLR) для VGD ===
lambda = 0.00001 | Train Loss: 108892256.44 | R2_train: -0.7892 | Val Loss: 106805268.34
lambda = 0.00010 | Train Loss: 105529637.54 | R2_train: -0.7339 | Val Loss: 103240731.30
lambda = 0.00100 | Train Loss:  79161378.08 | R2_train: -0.3007 | Val Loss:  75552644.08
lambda = 0.01000 | Train Loss:  19166981.61 | R2_train:  0.6851 | Val Loss:  17943280.88
lambda = 0.10000 | Train Loss:  12654639.88 | R2_train:  0.7921 | Val Loss:  17051391.26
lambda = 1.00000 | Train Loss:  12087551.55 | R2_train:  0.8014 | Val Loss:  17597803.58

--- Результаты VGD с TimeDecayLR ---
Лучший lambda:      0.1
Loss_train:         12654639.88
R^2_train:          0.7921
Loss_val:           17051391.26
Loss_test:          28993025.99
R^2_test:           0.5843
Число итераций:     1000


6. Исследуйте SGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [7]:
# Обучение Stochastic Gradient Descent (SGD) с постоянным шагом n
def train_sgd_const(X, y, eta, max_iter=1000, random_state=42):
    np.random.seed(random_state)
    l, d = X.shape
    w = np.zeros(d)
    iters = 0
    for k in range(1, max_iter + 1):
        iters = k
        # Случайно выбираем 1 объект (batch_size = 1)
        idx = np.random.randint(0, l)
        xi = X[idx:idx+1]
        yi = y[idx:idx+1]
        
        # Стохастический градиент
        grad = 2.0 * xi.T @ (xi @ w - yi)
        w -= eta * grad
        
        # Защита от расходимости в inf/nan
        if np.isnan(w).any() or np.isinf(w).any():
            break
            
    return w, iters

# Сетка шагов n от 10^-5 до 1
grid_n = np.logspace(-5, 0, 6)

best_loss_val_sgd = float('inf')
best_n_sgd = None
best_train_loss_sgd = None
best_train_r2_sgd = None

print("=== Перебор шага n для SGD ===")
for n in grid_n:
    w, iters = train_sgd_const(X_train, y_train, eta=n)
    
    pred_train = X_train @ w
    pred_val = X_val @ w
    
    loss_tr = compute_loss(y_train, pred_train)
    r2_tr = compute_r2(y_train, pred_train)
    loss_v = compute_loss(y_val, pred_val)
    
    print(f"n = {n:7.5f} | Train Loss: {loss_tr:12.2f} | R2_train: {r2_tr:7.4f} | Val Loss: {loss_v:12.2f}")
    
    # Отсекаем расходящиеся запуски (слишком большие потери или nan)
    if not np.isnan(loss_v) and not np.isinf(loss_v) and loss_v < 1e12:
        if loss_v < best_loss_val_sgd:
            best_loss_val_sgd = loss_v
            best_n_sgd = n
            best_train_loss_sgd = loss_tr
            best_train_r2_sgd = r2_tr

# Финальное тестирование на test с наилучшим n
best_w_sgd, test_iters_sgd = train_sgd_const(X_train, y_train, eta=best_n_sgd)
pred_test_sgd = X_test @ best_w_sgd
loss_test_sgd = compute_loss(y_test, pred_test_sgd)
r2_test_sgd = compute_r2(y_test, pred_test_sgd)

print("\n--- Результаты SGD с наилучшим постоянным шагом ---")
print(f"Лучший шаг n:       {best_n_sgd}")
print(f"Loss_train:         {best_train_loss_sgd:.2f}")
print(f"R^2_train:          {best_train_r2_sgd:.4f}")
print(f"Loss_val:           {best_loss_val_sgd:.2f}")
print(f"Loss_test:          {loss_test_sgd:.2f}")
print(f"R^2_test:           {r2_test_sgd:.4f}")
print(f"Число итераций:     {test_iters_sgd}")

=== Перебор шага n для SGD ===
n = 0.00001 | Train Loss: 103053555.89 | R2_train: -0.6932 | Val Loss: 100596454.49
n = 0.00010 | Train Loss:  66129429.49 | R2_train: -0.0865 | Val Loss:  62629739.40
n = 0.00100 | Train Loss:  42361618.21 | R2_train:  0.3040 | Val Loss:  22092644.95
n = 0.01000 | Train Loss: 3836952888902536073838592.00 | R2_train: -63043054390843792.0000 | Val Loss: 2626164820325489581228032.00
n = 0.10000 | Train Loss: 1526797071306138562545942609680189547943293085682295809787672542203374288997817231443037389660203288755385925605730458837887309341660073771653205780450481558392207230627982775802211216823768644714064953079989084065665094119723101112351206804267157682618844737319804555649411997771671317512192.00 | R2_train: -2508603926009238612489195211643716959677878786287657120436512707577061157416794355355051517739665941793355850226769339497419909375083030641180012639513027478397363091297191680826327254099189892513381099158510142078801498889576275468971038613255992179

C:\Users\kirill\AppData\Local\Temp\ipykernel_10680\476878569.py:15: RuntimeWarning: overflow encountered in matmul
  grad = 2.0 * xi.T @ (xi @ w - yi)
C:\Users\kirill\AppData\Local\Temp\ipykernel_10680\476878569.py:36: RuntimeWarning: overflow encountered in matmul
  pred_train = X_train @ w
C:\Users\kirill\AppData\Local\Temp\ipykernel_10680\476878569.py:36: RuntimeWarning: invalid value encountered in matmul
  pred_train = X_train @ w
C:\Users\kirill\AppData\Local\Temp\ipykernel_10680\476878569.py:37: RuntimeWarning: overflow encountered in matmul
  pred_val = X_val @ w
C:\Users\kirill\AppData\Local\Temp\ipykernel_10680\476878569.py:37: RuntimeWarning: invalid value encountered in matmul
  pred_val = X_val @ w


7. Исследуйте SGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [9]:
# Обучение SGD с переменным шагом TimeDecayLR
def train_sgd_decay(X, y, lam, max_iter=1000, s0=1.0, p=0.5, random_state=42):
    np.random.seed(random_state)
    l, d = X.shape
    w = np.zeros(d)
    iters = 0
    for k in range(1, max_iter + 1):
        iters = k
        # Формула шага из лекции
        eta_k = lam * ((s0 / (s0 + k)) ** p)
        
        idx = np.random.randint(0, l)
        xi = X[idx:idx+1]
        yi = y[idx:idx+1]
        
        grad = 2.0 * xi.T @ (xi @ w - yi)
        w -= eta_k * grad
        
        # Прерываем итерации при взрыве весов
        if np.isnan(w).any() or np.isinf(w).any():
            break
            
    return w, iters

# Сетка параметра lambda от 10^-5 до 1
grid_lambda = np.logspace(-5, 0, 6)

best_loss_val_sgd_decay = float('inf')
best_lam_sgd = None
best_train_loss_sgd_decay = None
best_train_r2_sgd_decay = None

print("=== Перебор параметра lambda (TimeDecayLR) для SGD ===")
for lam in grid_lambda:
    w, iters = train_sgd_decay(X_train, y_train, lam=lam)
    
    pred_train = X_train @ w
    pred_val = X_val @ w
    
    loss_tr = compute_loss(y_train, pred_train)
    r2_tr = compute_r2(y_train, pred_train)
    loss_v = compute_loss(y_val, pred_val)
    
    print(f"lambda = {lam:7.5f} | Train Loss: {loss_tr:12.2f} | R2_train: {r2_tr:7.4f} | Val Loss: {loss_v:12.2f}")
    
    # Фильтруем расходящиеся значения
    if not np.isnan(loss_v) and not np.isinf(loss_v) and loss_v < 1e12:
        if loss_v < best_loss_val_sgd_decay:
            best_loss_val_sgd_decay = loss_v
            best_lam_sgd = lam
            best_train_loss_sgd_decay = loss_tr
            best_train_r2_sgd_decay = r2_tr

# Финальное тестирование на test с наилучшим lambda
best_w_sgd_decay, test_iters_sgd_decay = train_sgd_decay(X_train, y_train, lam=best_lam_sgd)
pred_test_sgd_decay = X_test @ best_w_sgd_decay
loss_test_sgd_decay = compute_loss(y_test, pred_test_sgd_decay)
r2_test_sgd_decay = compute_r2(y_test, pred_test_sgd_decay)

print("\n--- Результаты SGD с TimeDecayLR ---")
print(f"Лучший lambda:      {best_lam_sgd}")
print(f"Loss_train:         {best_train_loss_sgd_decay:.2f}")
print(f"R^2_train:          {best_train_r2_sgd_decay:.4f}")
print(f"Loss_val:           {best_loss_val_sgd_decay:.2f}")
print(f"Loss_test:          {loss_test_sgd_decay:.2f}")
print(f"R^2_test:           {r2_test_sgd_decay:.4f}")
print(f"Число итераций:     {test_iters_sgd_decay}")

=== Перебор параметра lambda (TimeDecayLR) для SGD ===
lambda = 0.00001 | Train Loss: 108855964.72 | R2_train: -0.7886 | Val Loss: 106810282.11
lambda = 0.00010 | Train Loss: 105187747.11 | R2_train: -0.7283 | Val Loss: 103300208.41
lambda = 0.00100 | Train Loss:  77454583.02 | R2_train: -0.2726 | Val Loss:  76713469.20
lambda = 0.01000 | Train Loss:  91985053.31 | R2_train: -0.5114 | Val Loss:  24037608.87
lambda = 0.10000 | Train Loss: 46647487947162744.00 | R2_train: -766441549.1047 | Val Loss: 2966404833777648.00
lambda = 1.00000 | Train Loss: 5392555265236673179090769980488960283401837119581928382276812962116842470948059854141716970104303916011069236900282667178774418075303413748764786050909583991373824.00 | R2_train: -88602379214821317240668961545017038256665678676736857469505506665294554447501018607582823374929772970140324810003438567785689262315054430355040229180571648.0000 | Val Loss: 116423840493979360621847617202716323367275080264497406735897853384848892235478171695174271331

8. Исследуйте SAG с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [10]:
# Обучение SAG (Stochastic Average Gradient) с постоянным шагом n
def train_sag_const(X, y, eta, max_iter=1000, random_state=42):
    np.random.seed(random_state)
    n, d = X.shape
    w = np.zeros(d)
    
    # Инициализация массива сохраненных градиентов
    stored_grads = np.zeros((n, d))
    g_bar = np.zeros(d)
    
    iters = 0
    for k in range(1, max_iter + 1):
        iters = k
        # Случайный выбор объекта j
        j = np.random.randint(0, n)
        xj = X[j]
        yj = y[j]
        
        # Новый индивидуальный градиент по формуле из лекции
        g_new = 2.0 * (np.dot(xj, w) - yj) * xj
        
        # Обновление среднего градиента: \bar{g}_{k+1} = \bar{g}_k + 1/n * (g_j^{new} - g_j^{old})
        g_bar += (g_new - stored_grads[j]) / n
        stored_grads[j] = g_new
        
        # Обновление весов
        w -= eta * g_bar
        
        if np.isnan(w).any() or np.isinf(w).any():
            break
            
    return w, iters

# Сетка шагов n от 10^-5 до 1
grid_n = np.logspace(-5, 0, 6)

best_loss_val_sag = float('inf')
best_n_sag = None
best_train_loss_sag = None
best_train_r2_sag = None

print("=== Перебор шага n для SAG ===")
for n in grid_n:
    w, iters = train_sag_const(X_train, y_train, eta=n)
    
    pred_train = X_train @ w
    pred_val = X_val @ w
    
    loss_tr = compute_loss(y_train, pred_train)
    r2_tr = compute_r2(y_train, pred_train)
    loss_v = compute_loss(y_val, pred_val)
    
    print(f"n = {n:7.5f} | Train Loss: {loss_tr:12.2f} | R2_train: {r2_tr:7.4f} | Val Loss: {loss_v:12.2f}")
    
    # Отсекаем расходящиеся решения
    if not np.isnan(loss_v) and not np.isinf(loss_v) and loss_v < 1e12:
        if loss_v < best_loss_val_sag:
            best_loss_val_sag = loss_v
            best_n_sag = n
            best_train_loss_sag = loss_tr
            best_train_r2_sag = r2_tr

# Финальное тестирование на test с наилучшим n
best_w_sag, test_iters_sag = train_sag_const(X_train, y_train, eta=best_n_sag)
pred_test_sag = X_test @ best_w_sag
loss_test_sag = compute_loss(y_test, pred_test_sag)
r2_test_sag = compute_r2(y_test, pred_test_sag)

print("\n--- Результаты SAG с наилучшим постоянным шагом ---")
print(f"Лучший шаг n:       {best_n_sag}")
print(f"Loss_train:         {best_train_loss_sag:.2f}")
print(f"R^2_train:          {best_train_r2_sag:.4f}")
print(f"Loss_val:           {best_loss_val_sag:.2f}")
print(f"Loss_test:          {loss_test_sag:.2f}")
print(f"R^2_test:           {r2_test_sag:.4f}")
print(f"Число итераций:     {test_iters_sag}")

=== Перебор шага n для SAG ===
n = 0.00001 | Train Loss: 106469053.22 | R2_train: -0.7493 | Val Loss: 104601415.79
n = 0.00010 | Train Loss:  84852823.50 | R2_train: -0.3942 | Val Loss:  84262910.76
n = 0.00100 | Train Loss:  66428052.48 | R2_train: -0.0914 | Val Loss:  29105369.54
n = 0.01000 | Train Loss: 10738700422.69 | R2_train: -175.4422 | Val Loss: 482202182.51
n = 0.10000 | Train Loss: 1981458320472832212992.00 | R2_train: -32556350908568.0586 | Val Loss: 91888064095410585600.00
n = 1.00000 | Train Loss: 1983187031239184802969417820125871104966827102437376.00 | R2_train: -32584754490792964025250962237624396661391360.0000 | Val Loss: 250469782763258391919433262847118937979610032242688.00

--- Результаты SAG с наилучшим постоянным шагом ---
Лучший шаг n:       0.001
Loss_train:         66428052.48
R^2_train:          -0.0914
Loss_val:           29105369.54
Loss_test:          37311510.19
R^2_test:           0.4650
Число итераций:     1000


9. Исследуйте SAG с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [11]:
# Обучение SAG с переменным шагом TimeDecayLR
def train_sag_decay(X, y, lam, max_iter=1000, s0=1.0, p=0.5, random_state=42):
    np.random.seed(random_state)
    n, d = X.shape
    w = np.zeros(d)
    
    stored_grads = np.zeros((n, d))
    g_bar = np.zeros(d)
    
    iters = 0
    for k in range(1, max_iter + 1):
        iters = k
        # Формула шага из лекции
        eta_k = lam * ((s0 / (s0 + k)) ** p)
        
        j = np.random.randint(0, n)
        xj = X[j]
        yj = y[j]
        
        g_new = 2.0 * (np.dot(xj, w) - yj) * xj
        g_bar += (g_new - stored_grads[j]) / n
        stored_grads[j] = g_new
        
        w -= eta_k * g_bar
        
        if np.isnan(w).any() or np.isinf(w).any():
            break
            
    return w, iters

# Сетка параметра lambda от 10^-5 до 1
grid_lambda = np.logspace(-5, 0, 6)

best_loss_val_sag_decay = float('inf')
best_lam_sag = None
best_train_loss_sag_decay = None
best_train_r2_sag_decay = None

print("=== Перебор параметра lambda (TimeDecayLR) для SAG ===")
for lam in grid_lambda:
    w, iters = train_sag_decay(X_train, y_train, lam=lam)
    
    pred_train = X_train @ w
    pred_val = X_val @ w
    
    loss_tr = compute_loss(y_train, pred_train)
    r2_tr = compute_r2(y_train, pred_train)
    loss_v = compute_loss(y_val, pred_val)
    
    print(f"lambda = {lam:7.5f} | Train Loss: {loss_tr:12.2f} | R2_train: {r2_tr:7.4f} | Val Loss: {loss_v:12.2f}")
    
    if not np.isnan(loss_v) and not np.isinf(loss_v) and loss_v < 1e12:
        if loss_v < best_loss_val_sag_decay:
            best_loss_val_sag_decay = loss_v
            best_lam_sag = lam
            best_train_loss_sag_decay = loss_tr
            best_train_r2_sag_decay = r2_tr

# Финальное тестирование на test с наилучшим lambda
best_w_sag_decay, test_iters_sag_decay = train_sag_decay(X_train, y_train, lam=best_lam_sag)
pred_test_sag_decay = X_test @ best_w_sag_decay
loss_test_sag_decay = compute_loss(y_test, pred_test_sag_decay)
r2_test_sag_decay = compute_r2(y_test, pred_test_sag_decay)

print("\n--- Результаты SAG с TimeDecayLR ---")
print(f"Лучший lambda:      {best_lam_sag}")
print(f"Loss_train:         {best_train_loss_sag_decay:.2f}")
print(f"R^2_train:          {best_train_r2_sag_decay:.4f}")
print(f"Loss_val:           {best_loss_val_sag_decay:.2f}")
print(f"Loss_test:          {loss_test_sag_decay:.2f}")
print(f"R^2_test:           {r2_test_sag_decay:.4f}")
print(f"Число итераций:     {test_iters_sag_decay}")

=== Перебор параметра lambda (TimeDecayLR) для SAG ===
lambda = 0.00001 | Train Loss: 109146187.01 | R2_train: -0.7933 | Val Loss: 107095996.88
lambda = 0.00010 | Train Loss: 107987664.85 | R2_train: -0.7743 | Val Loss: 106056473.26
lambda = 0.00100 | Train Loss:  97228865.00 | R2_train: -0.5975 | Val Loss:  96328190.55
lambda = 0.01000 | Train Loss:  50121905.16 | R2_train:  0.1765 | Val Loss:  44377027.01
lambda = 0.10000 | Train Loss: 1539574781.29 | R2_train: -24.2960 | Val Loss:  47499053.08
lambda = 1.00000 | Train Loss: 23047877281966460.00 | R2_train: -378688146.3794 | Val Loss: 3280113496090766.00

--- Результаты SAG с TimeDecayLR ---
Лучший lambda:      0.01
Loss_train:         50121905.16
R^2_train:          0.1765
Loss_val:           44377027.01
Loss_test:          65417193.32
R^2_test:           0.0620
Число итераций:     1000


10. Исследуйте Momentum с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [12]:
# Обучение Momentum с постоянным шагом n
def train_momentum_const(X, y, eta, max_iter=1000, alpha=0.9, tol=1e-5):
    n_samples, d = X.shape
    w = np.zeros(d)
    h = np.zeros(d)
    iters = 0
    for k in range(1, max_iter + 1):
        iters = k
        w_prev = w.copy()
        
        # Полный градиент MSE
        grad = (2.0 / n_samples) * (X.T @ (X @ w - y))
        
        # Накопление импульса и обновление весов по формулам из лекции
        h = alpha * h + eta * grad
        w -= h
        
        if np.isnan(w).any() or np.isinf(w).any():
            break
        if np.linalg.norm(w - w_prev) < tol:
            break
            
    return w, iters

# Сетка шагов n от 10^-5 до 1
grid_n = np.logspace(-5, 0, 6)

best_loss_val_mom = float('inf')
best_n_mom = None
best_train_loss_mom = None
best_train_r2_mom = None

print("=== Перебор шага n для Momentum ===")
for n in grid_n:
    w, iters = train_momentum_const(X_train, y_train, eta=n)
    
    pred_train = X_train @ w
    pred_val = X_val @ w
    
    loss_tr = compute_loss(y_train, pred_train)
    r2_tr = compute_r2(y_train, pred_train)
    loss_v = compute_loss(y_val, pred_val)
    
    print(f"n = {n:7.5f} | Train Loss: {loss_tr:12.2f} | R2_train: {r2_tr:7.4f} | Val Loss: {loss_v:12.2f}")
    
    if not np.isnan(loss_v) and not np.isinf(loss_v):
        if loss_v < best_loss_val_mom:
            best_loss_val_mom = loss_v
            best_n_mom = n
            best_train_loss_mom = loss_tr
            best_train_r2_mom = r2_tr

# Финальное тестирование на test с наилучшим n
best_w_mom, test_iters_mom = train_momentum_const(X_train, y_train, eta=best_n_mom)
pred_test_mom = X_test @ best_w_mom
loss_test_mom = compute_loss(y_test, pred_test_mom)
r2_test_mom = compute_r2(y_test, pred_test_mom)

print("\n--- Результаты Momentum с наилучшим постоянным шагом ---")
print(f"Лучший шаг n:       {best_n_mom}")
print(f"Loss_train:         {best_train_loss_mom:.2f}")
print(f"R^2_train:          {best_train_r2_mom:.4f}")
print(f"Loss_val:           {best_loss_val_mom:.2f}")
print(f"Loss_test:          {loss_test_mom:.2f}")
print(f"R^2_test:           {r2_test_mom:.4f}")
print(f"Число итераций:     {test_iters_mom}")

=== Перебор шага n для Momentum ===
n = 0.00001 | Train Loss:  66308849.82 | R2_train: -0.0895 | Val Loss:  62308687.44
n = 0.00010 | Train Loss:  14669723.83 | R2_train:  0.7590 | Val Loss:  15256162.76
n = 0.00100 | Train Loss:  12509301.98 | R2_train:  0.7945 | Val Loss:  17337548.81
n = 0.01000 | Train Loss:  12044309.36 | R2_train:  0.8021 | Val Loss:  17792110.24
n = 0.10000 | Train Loss:  12028274.30 | R2_train:  0.8024 | Val Loss:  18428292.12
n = 1.00000 | Train Loss:          nan | R2_train:     nan | Val Loss:          nan

--- Результаты Momentum с наилучшим постоянным шагом ---
Лучший шаг n:       0.0001
Loss_train:         14669723.83
R^2_train:          0.7590
Loss_val:           15256162.76
Loss_test:          33169935.39
R^2_test:           0.5244
Число итераций:     1000


C:\Users\kirill\AppData\Local\Temp\ipykernel_10680\2815928655.py:12: RuntimeWarning: overflow encountered in matmul
  grad = (2.0 / n_samples) * (X.T @ (X @ w - y))
C:\Users\kirill\AppData\Local\Temp\ipykernel_10680\2815928655.py:37: RuntimeWarning: invalid value encountered in matmul
  pred_train = X_train @ w
C:\Users\kirill\AppData\Local\Temp\ipykernel_10680\2815928655.py:38: RuntimeWarning: invalid value encountered in matmul
  pred_val = X_val @ w


11. Исследуйте Momentum с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [13]:
# Обучение Momentum с переменным шагом TimeDecayLR
def train_momentum_decay(X, y, lam, max_iter=1000, alpha=0.9, s0=1.0, p=0.5, tol=1e-5):
    n_samples, d = X.shape
    w = np.zeros(d)
    h = np.zeros(d)
    iters = 0
    for k in range(1, max_iter + 1):
        iters = k
        w_prev = w.copy()
        
        # Шаг по формуле TimeDecayLR из лекции
        eta_k = lam * ((s0 / (s0 + k)) ** p)
        
        # Полный градиент MSE
        grad = (2.0 / n_samples) * (X.T @ (X @ w - y))
        
        # Накопление импульса с переменным шагом и обновление весов
        h = alpha * h + eta_k * grad
        w -= h
        
        if np.isnan(w).any() or np.isinf(w).any():
            break
        if np.linalg.norm(w - w_prev) < tol:
            break
            
    return w, iters

# Сетка параметра lambda от 10^-5 до 1
grid_lambda = np.logspace(-5, 0, 6)

best_loss_val_mom_decay = float('inf')
best_lam_mom = None
best_train_loss_mom_decay = None
best_train_r2_mom_decay = None

print("=== Перебор параметра lambda (TimeDecayLR) для Momentum ===")
for lam in grid_lambda:
    w, iters = train_momentum_decay(X_train, y_train, lam=lam)
    
    pred_train = X_train @ w
    pred_val = X_val @ w
    
    loss_tr = compute_loss(y_train, pred_train)
    r2_tr = compute_r2(y_train, pred_train)
    loss_v = compute_loss(y_val, pred_val)
    
    print(f"lambda = {lam:7.5f} | Train Loss: {loss_tr:12.2f} | R2_train: {r2_tr:7.4f} | Val Loss: {loss_v:12.2f}")
    
    if not np.isnan(loss_v) and not np.isinf(loss_v):
        if loss_v < best_loss_val_mom_decay:
            best_loss_val_mom_decay = loss_v
            best_lam_mom = lam
            best_train_loss_mom_decay = loss_tr
            best_train_r2_mom_decay = r2_tr

# Финальное тестирование на test с наилучшим lambda
best_w_mom_decay, test_iters_mom_decay = train_momentum_decay(X_train, y_train, lam=best_lam_mom)
pred_test_mom_decay = X_test @ best_w_mom_decay
loss_test_mom_decay = compute_loss(y_test, pred_test_mom_decay)
r2_test_mom_decay = compute_r2(y_test, pred_test_mom_decay)

print("\n--- Результаты Momentum с TimeDecayLR ---")
print(f"Лучший lambda:      {best_lam_mom}")
print(f"Loss_train:         {best_train_loss_mom_decay:.2f}")
print(f"R^2_train:          {best_train_r2_mom_decay:.4f}")
print(f"Loss_val:           {best_loss_val_mom_decay:.2f}")
print(f"Loss_test:          {loss_test_mom_decay:.2f}")
print(f"R^2_test:           {r2_test_mom_decay:.4f}")
print(f"Число итераций:     {test_iters_mom_decay}")

=== Перебор параметра lambda (TimeDecayLR) для Momentum ===
lambda = 0.00001 | Train Loss: 105545516.97 | R2_train: -0.7342 | Val Loss: 103257502.85
lambda = 0.00010 | Train Loss:  79198472.64 | R2_train: -0.3013 | Val Loss:  75587836.99
lambda = 0.00100 | Train Loss:  19001554.35 | R2_train:  0.6878 | Val Loss:  17803873.35
lambda = 0.01000 | Train Loss:  12653079.62 | R2_train:  0.7921 | Val Loss:  17070329.95
lambda = 0.10000 | Train Loss:  12086138.00 | R2_train:  0.8014 | Val Loss:  17600041.11
lambda = 1.00000 | Train Loss:  12028285.22 | R2_train:  0.8024 | Val Loss:  18414927.36

--- Результаты Momentum с TimeDecayLR ---
Лучший lambda:      0.01
Loss_train:         12653079.62
R^2_train:          0.7921
Loss_val:           17070329.95
Loss_test:          29004420.82
R^2_test:           0.5841
Число итераций:     1000


12. Исследуйте Adam с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [14]:
# Обучение Adam с постоянным шагом n
def train_adam_const(X, y, eta, max_iter=1000, beta1=0.9, beta2=0.999, eps=1e-8, tol=1e-5):
    n_samples, d = X.shape
    w = np.zeros(d)
    m = np.zeros(d)
    v = np.zeros(d)
    iters = 0
    for k in range(1, max_iter + 1):
        iters = k
        w_prev = w.copy()
        
        # Полный градиент MSE
        grad = (2.0 / n_samples) * (X.T @ (X @ w - y))
        
        # Обновление первого и второго моментов
        m = beta1 * m + (1.0 - beta1) * grad
        v = beta2 * v + (1.0 - beta2) * (grad ** 2)
        
        # Коррекция смещения
        m_hat = m / (1.0 - (beta1 ** k))
        v_hat = v / (1.0 - (beta2 ** k))
        
        # Обновление весов с постоянным шагом
        w -= eta * m_hat / (np.sqrt(v_hat) + eps)
        
        if np.isnan(w).any() or np.isinf(w).any():
            break
        if np.linalg.norm(w - w_prev) < tol:
            break
            
    return w, iters

# Сетка шагов n от 10^-5 до 1
grid_n = np.logspace(-5, 0, 6)

best_loss_val_adam = float('inf')
best_n_adam = None
best_train_loss_adam = None
best_train_r2_adam = None

print("=== Перебор шага n для Adam ===")
for n in grid_n:
    w, iters = train_adam_const(X_train, y_train, eta=n)
    
    pred_train = X_train @ w
    pred_val = X_val @ w
    
    loss_tr = compute_loss(y_train, pred_train)
    r2_tr = compute_r2(y_train, pred_train)
    loss_v = compute_loss(y_val, pred_val)
    
    print(f"n = {n:7.5f} | Train Loss: {loss_tr:12.2f} | R2_train: {r2_tr:7.4f} | Val Loss: {loss_v:12.2f}")
    
    if not np.isnan(loss_v) and not np.isinf(loss_v):
        if loss_v < best_loss_val_adam:
            best_loss_val_adam = loss_v
            best_n_adam = n
            best_train_loss_adam = loss_tr
            best_train_r2_adam = r2_tr

# Финальное тестирование на test с наилучшим n
best_w_adam, test_iters_adam = train_adam_const(X_train, y_train, eta=best_n_adam)
pred_test_adam = X_test @ best_w_adam
loss_test_adam = compute_loss(y_test, pred_test_adam)
r2_test_adam = compute_r2(y_test, pred_test_adam)

print("\n--- Результаты Adam с наилучшим постоянным шагом ---")
print(f"Лучший шаг n:       {best_n_adam}")
print(f"Loss_train:         {best_train_loss_adam:.2f}")
print(f"R^2_train:          {best_train_r2_adam:.4f}")
print(f"Loss_val:           {best_loss_val_adam:.2f}")
print(f"Loss_test:          {loss_test_adam:.2f}")
print(f"R^2_test:           {r2_test_adam:.4f}")
print(f"Число итераций:     {test_iters_adam}")

=== Перебор шага n для Adam ===
n = 0.00001 | Train Loss: 109274069.62 | R2_train: -0.7954 | Val Loss: 107210179.39
n = 0.00010 | Train Loss: 109257857.01 | R2_train: -0.7952 | Val Loss: 107191285.70
n = 0.00100 | Train Loss: 109096102.65 | R2_train: -0.7925 | Val Loss: 107002699.84
n = 0.01000 | Train Loss: 107514918.48 | R2_train: -0.7665 | Val Loss: 105156823.61
n = 0.10000 | Train Loss:  94723911.16 | R2_train: -0.5564 | Val Loss:  90242221.60
n = 1.00000 | Train Loss:  57146017.14 | R2_train:  0.0611 | Val Loss:  50385916.88

--- Результаты Adam с наилучшим постоянным шагом ---
Лучший шаг n:       1.0
Loss_train:         57146017.14
R^2_train:          0.0611
Loss_val:           50385916.88
Loss_test:          96817210.82
R^2_test:           -0.3883
Число итераций:     1000


13. Исследуйте Adam с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [16]:
# Обучение Adam с переменным шагом TimeDecayLR
def train_adam_decay(X, y, lam, max_iter=1000, beta1=0.9, beta2=0.999, eps=1e-8, s0=1.0, p=0.5, tol=1e-5):
    n_samples, d = X.shape
    w = np.zeros(d)
    m = np.zeros(d)
    v = np.zeros(d)
    iters = 0
    for k in range(1, max_iter + 1):
        iters = k
        w_prev = w.copy()
        
        # Переменный шаг по формуле из лекции
        eta_k = lam * ((s0 / (s0 + k)) ** p)
        
        grad = (2.0 / n_samples) * (X.T @ (X @ w - y))
        m = beta1 * m + (1.0 - beta1) * grad
        v = beta2 * v + (1.0 - beta2) * (grad ** 2)
        
        m_hat = m / (1.0 - (beta1 ** k))
        v_hat = v / (1.0 - (beta2 ** k))
        
        w -= eta_k * m_hat / (np.sqrt(v_hat) + eps)
        
        if np.isnan(w).any() or np.isinf(w).any():
            break
        if np.linalg.norm(w - w_prev) < tol:
            break
            
    return w, iters

# Сетка параметра lambda от 10^-5 до 1
grid_lambda = np.logspace(-5, 0, 6)

best_loss_val_adam_decay = float('inf')
best_lam_adam = None
best_train_loss_adam_decay = None
best_train_r2_adam_decay = None

print("=== Перебор параметра lambda (TimeDecayLR) для Adam ===")
for lam in grid_lambda:
    w, iters = train_adam_decay(X_train, y_train, lam=lam)
    
    pred_train = X_train @ w
    pred_val = X_val @ w
    
    loss_tr = compute_loss(y_train, pred_train)
    r2_tr = compute_r2(y_train, pred_train)
    loss_v = compute_loss(y_val, pred_val)
    
    print(f"lambda = {lam:7.5f} | Train Loss: {loss_tr:12.2f} | R2_train: {r2_tr:7.4f} | Val Loss: {loss_v:12.2f}")
    
    if not np.isnan(loss_v) and not np.isinf(loss_v):
        if loss_v < best_loss_val_adam_decay:
            best_loss_val_adam_decay = loss_v
            best_lam_adam = lam
            best_train_loss_adam_decay = loss_tr
            best_train_r2_adam_decay = r2_tr

# Финальное тестирование на test с наилучшим lambda
best_w_adam_decay, test_iters_adam_decay = train_adam_decay(X_train, y_train, lam=best_lam_adam)
pred_test_adam_decay = X_test @ best_w_adam_decay
loss_test_adam_decay = compute_loss(y_test, pred_test_adam_decay)
r2_test_adam_decay = compute_r2(y_test, pred_test_adam_decay)

print("\n--- Результаты Adam с TimeDecayLR ---")
print(f"Лучший lambda:      {best_lam_adam}")
print(f"Loss_train:         {best_train_loss_adam_decay:.2f}")
print(f"R^2_train:          {best_train_r2_adam_decay:.4f}")
print(f"Loss_val:           {best_loss_val_adam_decay:.2f}")
print(f"Loss_test:          {loss_test_adam_decay:.2f}")
print(f"R^2_test:           {r2_test_adam_decay:.4f}")
print(f"Число итераций:     {test_iters_adam_decay}")

=== Перебор параметра lambda (TimeDecayLR) для Adam ===
lambda = 0.00001 | Train Loss: 109275826.26 | R2_train: -0.7955 | Val Loss: 107212226.46
lambda = 0.00010 | Train Loss: 109274775.33 | R2_train: -0.7954 | Val Loss: 107211001.78
lambda = 0.00100 | Train Loss: 109264911.66 | R2_train: -0.7953 | Val Loss: 107199507.16
lambda = 0.01000 | Train Loss: 109166400.09 | R2_train: -0.7937 | Val Loss: 107084682.60
lambda = 0.10000 | Train Loss: 108193672.25 | R2_train: -0.7777 | Val Loss: 105948516.53
lambda = 1.00000 | Train Loss:  99598585.41 | R2_train: -0.6365 | Val Loss:  95879806.88

--- Результаты Adam с TimeDecayLR ---
Лучший lambda:      1.0
Loss_train:         99598585.41
R^2_train:          -0.6365
Loss_val:           95879806.88
Loss_test:          125195682.11
R^2_test:           -0.7952
Число итераций:     1000


14. Постройте итоговую сравнительную таблицу со следующими столбцами:

1) название метода

2) значение лучшего шага (если n) или функция лучшего шага (если n(lyamda))

3) Loss_train

4) Loss_test

5) R^2 train

6) R^2 test

7) число итераций на test

In [17]:
summary_data = [
    {
        "Название метода": "VGD (const)",
        "Лучший шаг / функция шага": f"n = {best_n_vgd}",
        "Loss_train": round(best_train_loss_vgd, 2),
        "Loss_test": round(loss_test_vgd, 2),
        "R^2 train": round(best_train_r2_vgd, 4),
        "R^2 test": round(r2_test_vgd, 4),
        "Число итераций на test": test_iters_vgd
    },
    {
        "Название метода": "VGD (decay)",
        "Лучший шаг / функция шага": f"n(k) = {best_lam_vgd} / sqrt(1 + k)",
        "Loss_train": round(best_train_loss_vgd_decay, 2),
        "Loss_test": round(loss_test_vgd_decay, 2),
        "R^2 train": round(best_train_r2_vgd_decay, 4),
        "R^2 test": round(r2_test_vgd_decay, 4),
        "Число итераций на test": test_iters_vgd_decay
    },
    {
        "Название метода": "SGD (const)",
        "Лучший шаг / функция шага": f"n = {best_n_sgd}",
        "Loss_train": round(best_train_loss_sgd, 2),
        "Loss_test": round(loss_test_sgd, 2),
        "R^2 train": round(best_train_r2_sgd, 4),
        "R^2 test": round(r2_test_sgd, 4),
        "Число итераций на test": test_iters_sgd
    },
    {
        "Название метода": "SGD (decay)",
        "Лучший шаг / функция шага": f"n(k) = {best_lam_sgd} / sqrt(1 + k)",
        "Loss_train": round(best_train_loss_sgd_decay, 2),
        "Loss_test": round(loss_test_sgd_decay, 2),
        "R^2 train": round(best_train_r2_sgd_decay, 4),
        "R^2 test": round(r2_test_sgd_decay, 4),
        "Число итераций на test": test_iters_sgd_decay
    },
    {
        "Название метода": "SAG (const)",
        "Лучший шаг / функция шага": f"n = {best_n_sag}",
        "Loss_train": round(best_train_loss_sag, 2),
        "Loss_test": round(loss_test_sag, 2),
        "R^2 train": round(best_train_r2_sag, 4),
        "R^2 test": round(r2_test_sag, 4),
        "Число итераций на test": test_iters_sag
    },
    {
        "Название метода": "SAG (decay)",
        "Лучший шаг / функция шага": f"n(k) = {best_lam_sag} / sqrt(1 + k)",
        "Loss_train": round(best_train_loss_sag_decay, 2),
        "Loss_test": round(loss_test_sag_decay, 2),
        "R^2 train": round(best_train_r2_sag_decay, 4),
        "R^2 test": round(r2_test_sag_decay, 4),
        "Число итераций на test": test_iters_sag_decay
    },
    {
        "Название метода": "Momentum (const)",
        "Лучший шаг / функция шага": f"n = {best_n_mom}",
        "Loss_train": round(best_train_loss_mom, 2),
        "Loss_test": round(loss_test_mom, 2),
        "R^2 train": round(best_train_r2_mom, 4),
        "R^2 test": round(r2_test_mom, 4),
        "Число итераций на test": test_iters_mom
    },
    {
        "Название метода": "Momentum (decay)",
        "Лучший шаг / функция шага": f"n(k) = {best_lam_mom} / sqrt(1 + k)",
        "Loss_train": round(best_train_loss_mom_decay, 2),
        "Loss_test": round(loss_test_mom_decay, 2),
        "R^2 train": round(best_train_r2_mom_decay, 4),
        "R^2 test": round(r2_test_mom_decay, 4),
        "Число итераций на test": test_iters_mom_decay
    },
    {
        "Название метода": "Adam (const)",
        "Лучший шаг / функция шага": f"n = {best_n_adam}",
        "Loss_train": round(best_train_loss_adam, 2),
        "Loss_test": round(loss_test_adam, 2),
        "R^2 train": round(best_train_r2_adam, 4),
        "R^2 test": round(r2_test_adam, 4),
        "Число итераций на test": test_iters_adam
    },
    {
        "Название метода": "Adam (decay)",
        "Лучший шаг / функция шага": f"n(k) = {best_lam_adam} / sqrt(1 + k)",
        "Loss_train": round(best_train_loss_adam_decay, 2),
        "Loss_test": round(loss_test_adam_decay, 2),
        "R^2 train": round(best_train_r2_adam_decay, 4),
        "R^2 test": round(r2_test_adam_decay, 4),
        "Число итераций на test": test_iters_adam_decay
    }
]

df_summary = pd.DataFrame(summary_data)
display(df_summary)

,Название метода,Лучший шаг / функция шага,Loss_train,Loss_test,R^2 train,R^2 test,Число итераций на test
0,VGD (const),n = 0.001,14705040.17,3.319693e+07,0.7584,0.5240,1000
1,VGD (decay),n(k) = 0.1 / sqrt(1 + k),12654639.88,2.899303e+07,0.7921,0.5843,1000
2,SGD (const),n = 0.001,42361618.21,2.528184e+07,0.3040,0.6375,1000
3,SGD (decay),n(k) = 0.01 / sqrt(1 + k),91985053.31,3.471461e+07,-0.5114,0.5022,1000
4,SAG (const),n = 0.001,66428052.48,3.731151e+07,-0.0914,0.4650,1000
5,SAG (decay),n(k) = 0.01 / sqrt(1 + k),50121905.16,6.541719e+07,0.1765,0.0620,1000
6,Momentum (const),n = 0.0001,14669723.83,3.316994e+07,0.7590,0.5244,1000
7,Momentum (decay),n(k) = 0.01 / sqrt(1 + k),12653079.62,2.900442e+07,0.7921,0.5841,1000
8,Adam (const),n = 1.0,57146017.14,9.681721e+07,0.0611,-0.3883,1000
9,Adam (decay),n(k) = 1.0 / sqrt(1 + k),99598585.41,1.251957e+08,-0.6365,-0.7952,1000


15. Сделайте вывод о том, какой метод и шаг линейной регрессии самый лучший для данной выборки и ответьте на вопросы:

1) почему именно этот метод и этот шаг самый лучший (по каким данным из таблицы вы сделали такой вывод)

2) расскажите простыми словами суть R^2_train и R^2_test?

3) как R^2_train и R^2_test помогают сравнивать методы? почему оба эти значения надо вычислять для данного выбора?

In [ ]:
1. Самым лучшим вышел SGD (const) с шагом n = 0.001. Это понятно по столбцу loss test (величина ошибки), она самая наименьшая именно у SGD (const). И ещё это понятно по столбцу R^2 test, там за также самое большое
2. Само число R^2 означает, насколько процентов модель отвечает лучше, чем простое предсказание константы среднего значение цен. train это на тренировочной выборки. А test соответственно на тестовой
3. Эти две метрики помогают понять, обучилась ли модель, переобучилась или наоборот недоучилась. Если R^2 train высокий, а test низкий, значит модель переобучилась. Если оба около нуля или даже отрицательные, то модель недообучилась. Если же оба высокие и близки друг к другу, то модель обучилась